In [ ]:

"""
train_landmark_lstm.py

Usage:
    python train_landmark_lstm.py --clips_dir ./clips --ann_csv ./annotations.csv --out_dir ./sequences --labels labels.txt --seq_len 32 --epochs 20 --batch 16

What it does:
- Extracts MediaPipe Holistic landmarks from each clip and saves compressed .npz sequences
- Builds a PyTorch Dataset and DataLoader from the .npz files
- Trains a simple LSTM classifier
- Saves best model to best_lstm_model.pth and class prototypes to class_prototypes.npz
"""

import argparse
import os
from pathlib import Path
import csv
import glob
import numpy as np
import cv2
from tqdm import tqdm
import mediapipe as mp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
import collections

mp_holistic = mp.solutions.holistic

# ---------- Configurable functions ----------
def extract_landmarks_from_video(video_path, seq_len=32, fps_target=25, use_face=False, use_pose=True, use_hands=True):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print("Could not open", video_path)
        return None
    orig_fps = cap.get(cv2.CAP_PROP_FPS) or fps_target
    frame_step = max(1, int(round(orig_fps / fps_target)))
    with mp_holistic.Holistic(static_image_mode=False, model_complexity=1) as holistic:
        frames = []
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx % frame_step != 0:
                idx += 1
                continue
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            feat = []
            # left hand 21*3
            if use_hands:
                if results.left_hand_landmarks:
                    for lm in results.left_hand_landmarks.landmark:
                        feat.extend([lm.x, lm.y, lm.z])
                else:
                    feat.extend([0.0] * 21 * 3)
                if results.right_hand_landmarks:
                    for lm in results.right_hand_landmarks.landmark:
                        feat.extend([lm.x, lm.y, lm.z])
                else:
                    feat.extend([0.0] * 21 * 3)
            # pose 33*3
            if use_pose:
                if results.pose_landmarks:
                    for lm in results.pose_landmarks.landmark:
                        feat.extend([lm.x, lm.y, lm.z])
                else:
                    feat.extend([0.0] * 33 * 3)
            # face optional (skip by default)
            if use_face:
                if results.face_landmarks:
                    for lm in results.face_landmarks.landmark:
                        feat.extend([lm.x, lm.y, lm.z])
                else:
                    feat.extend([0.0] * 468 * 3)
            frames.append(feat)
            idx += 1
    cap.release()
    frames = np.array(frames)
    if frames.shape[0] == 0:
        return None
    # pad or trim
    if frames.shape[0] < seq_len:
        pad = np.zeros((seq_len - frames.shape[0], frames.shape[1]))
        frames = np.vstack([frames, pad])
    else:
        frames = frames[:seq_len]
    return frames

class LandmarkSequenceDataset(Dataset):
    def __init__(self, seq_dir):
        self.files = sorted(glob.glob(str(Path(seq_dir) / '*.npz')))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        arr = data['arr'].astype(np.float32)
        label = int(data['label'])
        # simple normalization: zero mean per sample
        arr = arr - np.mean(arr, axis=0)
        return torch.from_numpy(arr), label

class SignLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=100, num_layers=2):
        super(SignLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        h_last = h[-1]
        return self.fc(h_last)

def build_sequences(clips_dir, ann_csv, out_dir, seq_len, fps):
    os.makedirs(out_dir, exist_ok=True)
    with open(ann_csv, 'r') as f:
        reader = csv.reader(f)
        entries = list(reader)
    for fname, label in tqdm(entries):
        video_path = Path(clips_dir) / fname
        if not video_path.exists():
            print("Missing", video_path)
            continue
        arr = extract_landmarks_from_video(video_path, seq_len=seq_len, fps_target=fps)
        if arr is None:
            print("No frames for", fname)
            continue
        out_path = Path(out_dir) / (Path(fname).stem + '.npz')
        np.savez_compressed(out_path, arr=arr, label=int(label))

def compute_class_prototypes(seq_dir, out_path='class_prototypes.npz'):
    files = sorted(glob.glob(str(Path(seq_dir) / '*.npz')))
    accum = collections.defaultdict(list)
    for f in files:
        d = np.load(f)
        arr = d['arr']
        label = int(d['label'])
        accum[label].append(arr)
    prototypes = {}
    for label, arrs in accum.items():
        prototypes[label] = np.mean(np.stack(arrs), axis=0)
    # save as dict with string keys
    np.savez_compressed(out_path, **{str(k):v for k,v in prototypes.items()})
    print("Saved prototypes to", out_path)

def train(seq_dir, num_classes, epochs, batch_size, lr, device):
    dataset = LandmarkSequenceDataset(seq_dir)
    n = len(dataset)
    if n == 0:
        print("No sequence files found. Run build_sequences first.")
        return
    train_n = int(n * 0.8)
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_n, n-train_n])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    sample_x, _ = dataset[0]
    input_dim = sample_x.shape[1]
    model = SignLSTM(input_dim=input_dim, hidden_dim=256, num_classes=num_classes).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0
    for epoch in range(epochs):
        model.train()
        losses = []
        for X, y in train_loader:
            X = X.to(device).float()
            y = y.to(device).long()
            opt.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        # val
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv = Xv.to(device).float()
                out = model(Xv)
                p = out.argmax(dim=1).cpu().numpy()
                preds.extend(p.tolist())
                trues.extend(yv.numpy().tolist())
        acc = accuracy_score(trues, preds) if len(trues)>0 else 0.0
        print(f"Epoch {epoch+1}/{epochs} loss={np.mean(losses):.4f} val_acc={acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), 'best_lstm_model.pth')
    print("Training complete. Best val acc:", best_acc)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--clips_dir', type=str, required=True)
    parser.add_argument('--ann_csv', type=str, required=True)
    parser.add_argument('--out_dir', type=str, default='./sequences')
    parser.add_argument('--seq_len', type=int, default=32)
    parser.add_argument('--fps', type=int, default=25)
    parser.add_argument('--epochs', type=int, default=20)
    parser.add_argument('--batch', type=int, default=16)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--num_classes', type=int, required=True)
    parser.add_argument('--device', type=str, default='cpu')
    args = parser.parse_args()

    # Step 1: build sequences (if not already)
    print("Building sequences ...")
    build_sequences(args.clips_dir, args.ann_csv, args.out_dir, args.seq_len, args.fps)

    # Step 2: train
    print("Training ...")
    train(args.out_dir, args.num_classes, args.epochs, args.batch, args.lr, args.device)

    # Step 3: compute prototypes
    print("Computing class prototypes ...")
    compute_class_prototypes(args.out_dir, out_path='class_prototypes.npz')
    print("Done. Saved model 'best_lstm_model.pth' and 'class_prototypes.npz'.")

if __name__ == '__main__':
    main()


In [1]:
import os
import cv2
import yaml
import glob
import numpy as np
import torch
import mediapipe as mp
from ultralytics import YOLO

# === Load class names from data.yaml ===
with open(r"C:\Users\kusal\OneDrive\Documents\Uni stuff\Year 5 Sem 2\FYP-B\asl-dataset-p9yw8 v2\data.yaml", "r") as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml['names']

# === Collect one image per class from train/valid/test folders ===
image_per_class = {}
for split in ['train', 'valid', 'test']:
    if split not in data_yaml:
        continue
    images_dir = data_yaml[split]
    label_dir = images_dir.replace('images', 'labels')
    image_files = glob.glob(os.path.join(images_dir, "*.jpg")) + glob.glob(os.path.join(images_dir, "*.png"))
    for img_path in image_files:
        label_path = os.path.join(label_dir, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
        if os.path.exists(label_path):
            with open(label_path, "r") as lf:
                for line in lf:
                    class_id = int(line.split()[0])
                    if class_id not in image_per_class:
                        image_per_class[class_id] = img_path
                    if len(image_per_class) == len(class_names):
                        break
    if len(image_per_class) == len(class_names):
        break

# === Prepare reference list ===
reference_list = [(image_per_class[c], class_names[c]) for c in sorted(image_per_class.keys())]

# === Load YOLO model ===
model = YOLO("runs/detect/asl-dataset-p9yw8 v2/weights/best.pt")

# === Init MediaPipe Holistic ===
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, model_complexity=1)

# === Optional: load trained LSTM model for landmark sequences ===
class SignLSTM(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=100, num_layers=2):
        super(SignLSTM, self).__init__()
        self.lstm = torch.nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.3)
        self.fc = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        out, (h,c) = self.lstm(x)
        h_last = h[-1]
        return self.fc(h_last)

def load_lstm(model_path, input_dim, num_classes, device):
    m = SignLSTM(input_dim=input_dim, num_classes=num_classes)
    if os.path.exists(model_path):
        m.load_state_dict(torch.load(model_path, map_location=device))
        print("✅ Loaded LSTM model")
    else:
        print("⚠️ No LSTM model found, skipping sequence classification")
    m.to(device).eval()
    return m

# === Configure LSTM classifier ===
device = "cuda" if torch.cuda.is_available() else "cpu"
SEQ_LEN = 32
INPUT_DIM = (21*3)*2 + (33*3)  # hands + pose
NUM_CLASSES = len(class_names)
lstm_model = load_lstm("best_lstm_model.pth", INPUT_DIM, NUM_CLASSES, device)
sequence_buffer = []

# === Extract landmarks into vector ===
def extract_feat(results):
    feat = []
    # Left hand
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            feat.extend([lm.x, lm.y, lm.z])
    else:
        feat.extend([0.0]*21*3)
    # Right hand
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            feat.extend([lm.x, lm.y, lm.z])
    else:
        feat.extend([0.0]*21*3)
    # Pose (optional)
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            feat.extend([lm.x, lm.y, lm.z])
    else:
        feat.extend([0.0]*33*3)
    return np.array(feat, dtype=np.float32)

# === Start webcam ===
cap = cv2.VideoCapture(0)
idx = 0
scale = 1.5  # scale output window

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # === Run YOLO detection ===
    results = model(frame, imgsz=640, conf=0.5)
    annotated_frame = results[0].plot()

    # === Run MediaPipe inside YOLO boxes ===
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        img_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        mp_res = holistic.process(img_rgb)

        feat = extract_feat(mp_res)
        sequence_buffer.append(feat)
        if len(sequence_buffer) > SEQ_LEN:
            sequence_buffer.pop(0)

        # === Predict with LSTM if buffer full ===
        if len(sequence_buffer) == SEQ_LEN:
            x = np.stack(sequence_buffer).astype(np.float32)
            x = x - np.mean(x, axis=0)  # normalize
            inp = torch.from_numpy(x).unsqueeze(0).to(device)
            with torch.no_grad():
                out = lstm_model(inp)
                pred = int(torch.argmax(out, dim=1).item())
            pred_label = class_names[pred] if pred < len(class_names) else str(pred)
            cv2.putText(annotated_frame, f"LSTM: {pred_label}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)

    # === Add reference image ===
    h, w, _ = annotated_frame.shape
    reference_image_path, reference_label = reference_list[idx]
    reference_image = cv2.imread(reference_image_path)
    ref_aspect = reference_image.shape[1] / reference_image.shape[0]
    reference_image = cv2.resize(reference_image, (int(h * ref_aspect), h))
    ref_img_with_label = reference_image.copy()
    cv2.putText(ref_img_with_label, f"Sign: {reference_label}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    # === Combine reference + webcam ===
    combined = cv2.hconcat([ref_img_with_label, annotated_frame])
    if scale != 1.0:
        combined = cv2.resize(combined, (0,0), fx=scale, fy=scale)

    cv2.imshow("Reference (Left) | Webcam + YOLO + LSTM (Right)", combined)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key in [13, 32]:  # Enter/Space to change reference
        idx = (idx + 1) % len(reference_list)

cap.release()
cv2.destroyAllWindows()

⚠️ No LSTM model found, skipping sequence classification

0: 480x640 (no detections), 70.1ms
Speed: 4.8ms preprocess, 70.1ms inference, 28.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 8.7ms
Speed: 3.1ms preprocess, 8.7ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 6.0ms
Speed: 1.1ms preprocess, 6.0ms inference, 5.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 10.4ms
Speed: 2.3ms preprocess, 10.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 10.5ms
Speed: 1.8ms preprocess, 10.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 26.9ms
Speed: 2.1ms preprocess, 26.9ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 16.7ms
Speed: 1.0ms preprocess, 16.7ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no